In [14]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [21]:
# データの準備
df = sns.load_dataset('titanic')
df.dropna(inplace=True)

#X, yを作成
X = df.loc[:, (df.columns!='survived') & (df.columns!='alive')]
y = df['survived']
oe = OrdinalEncoder()
oe.set_output(transform='pandas')
X = oe.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)

In [7]:
for train_idx, test_idx in cv.split(X_train):
    print(train_idx, test_idx)

[  0   1   3   4   5   6   9  11  12  13  14  15  17  18  19  20  21  23
  25  26  27  28  29  31  32  33  34  35  36  37  38  39  41  42  43  44
  45  46  47  49  50  52  53  54  55  56  57  58  60  61  62  63  64  65
  67  68  69  70  72  74  75  76  77  79  80  81  82  83  84  86  87  88
  90  93  94  96  97  99 102 103 104 106 107 108 109 110 111 112 113 114
 115 116 117 118 119 120 121 122 123 124 125] [  2   7   8  10  16  22  24  30  40  48  51  59  66  71  73  78  85  89
  91  92  95  98 100 101 105 126]
[  0   1   2   4   5   7   8   9  10  12  14  15  16  17  19  20  21  22
  23  24  25  28  29  30  31  32  34  35  36  37  38  39  40  41  42  44
  46  47  48  49  51  53  55  56  57  58  59  61  64  65  66  67  69  70
  71  72  73  74  76  77  78  79  80  81  82  83  85  86  87  88  89  90
  91  92  93  95  97  98  99 100 101 102 103 105 106 108 109 111 112 113
 114 115 116 117 118 119 120 122 123 125 126] [  3   6  11  13  18  26  27  33  43  45  50  52  54  60  62  63  68  7

In [26]:
class StackingClassifierCV:
    def __init__(self, estimators, final_estimator, cv):
        # estimator 1層目のモデルのリスト
        # final_estimator　2層目のsklearnモデルインスタンス
        # cv sklearnのCVオブジェクト
        self.estimators = estimators # [('rf', RandomForest()), ('knn', KneiborsClassifier),() ...]
        self.final_estimator = final_estimator
        self.cv = cv
        
    def fit(self, X, y):
        pred_features = {}
        # 1層目のモデル学習
        for model_name, model in self.estimators:
            preds = []
            new_y = [] # predはmodelで処理したときに順番が変わるので、同じ順番のyを用意する
            
            for train_idx, val_idx in cv.split(X):
                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
                model.fit(X_train, y_train)
                pred = model.predict_proba(X_val)[:, 1].tolist() # 2列で返ってくる
                preds += pred
                new_y += y_val.tolist()
            model.fit(X, y)
            pred_features[model_name] = preds # イテレーション毎にpredsが初期化してしまうから
        
        # 2層目の学習モデル
        new_X = pd.DataFrame(pred_features)
        self.final_estimator.fit(new_X, new_y)
        
    def predict_proba(self, X):
        # 1層目のモデルで特徴量（予測値）生成
        pred_features = {}
        for model_name, model in self.estimators:
            pred = model.predict_proba(X)[:, 1]
            pred_features[model_name] = pred
            
        new_X = pd.DataFrame(pred_features)
        final_pred = self.final_estimator.predict_proba(new_X)
        return final_pred
            
        
        

In [28]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
final_estimator = LogisticRegression()
stacking_cv = StackingClassifierCV(estimators=[('rf', RandomForestClassifier()), ('knn', KNeighborsClassifier())],
                                 final_estimator=final_estimator,
                                 cv=cv)
stacking_cv.fit(X_train, y_train)
y_pred_stacking_cv = stacking_cv.predict_proba(X_test)

In [31]:
print(f"stackingCV AUC: {roc_auc_score(y_test, y_pred_stacking_cv[:, 1])}")

stackingCV AUC: 0.8229166666666666
